# NIH Tracker guided review workbench (v19.2.2; extends v19.2.1)

The normal workflow is now **Review → Approve → Finalize**. The notebook shows only the next material action and skips decision families that have no pending work. Evidence-gap inventories and historical queues are available under **Advanced diagnostics**; they are not mixed into the operational review path and do not block publication.

Model output remains a recommendation. A named human reviewer must approve a recommendation before it can enter an operational adjudication file. Existing card, identifier, and **Crosswalk references** controls, locking, snapshots, and rollback safeguards remain in force.


## 1. Start the workbench

The workbench is production-only. Provide the reviewer identity through the `NIH_TRACKER_REVIEWER` Colab secret. Enable production writes only for the final **Apply all reviewed decisions** action.


In [ ]:
NOTEBOOK_NAME = "review_workbench"

ENVIRONMENT = "prod"
ENABLE_PROD_WRITES = False  # @param {type:"boolean"}

import os
import sys
from pathlib import Path

ROOTS = {"prod": "/content/drive/MyDrive/NIH_Tracker"}
os.environ["NIH_TRACKER_PRODUCTION_ONLY"] = "1"
os.environ["NIH_TRACKER_ROOT_PROD"] = ROOTS["prod"]
os.environ["NIH_TRACKER_ENV"] = "prod"
os.environ["NIH_TRACKER_SHARED_CACHE_ROOT"] = "/content/drive/MyDrive/NIH_Tracker_shared/cache"
os.environ.pop("NIH_TRACKER", None)
os.environ.pop("NIH_TRACKER_ALLOW_PROJECT_OVERRIDE", None)
os.environ.pop("NIH_TRACKER_ROOT_DEV", None)
os.environ.pop("NIH_TRACKER_ROOT_STAGING", None)

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

SCRIPTS = Path(ROOTS["prod"]) / "scripts"
if not SCRIPTS.is_dir():
    raise FileNotFoundError(
        f"Missing {SCRIPTS}. Run the maintenance notebook's production release installer first."
    )
# NIH_TRACKER_INSTALLED_RELEASE_CHECK
globals().pop("RUNTIME", None)
import hashlib
import json
def _check_installed_release(project, expected):
    manifest_path = project / "RELEASE_MANIFEST.json"
    guidance = "Open the matching maintenance notebook, preview step 1, set APPLY_PREFILL=True and rerun step 1. After successful installation, restart the runtime and run setup again."
    if not manifest_path.is_file():
        raise RuntimeError("Installed release manifest is missing. " + guidance)
    manifest = json.loads(manifest_path.read_text())
    if manifest.get("release") != expected:
        raise RuntimeError(f"Notebook requires {expected}; Drive has {manifest.get('release', 'unknown')}. " + guidance)
    critical = ("common.py", "notebook_runtime.py", "run_provenance.py", "code_provenance.py",
                "step_initialize_environments.py", "step_maintenance.py", "maintenance_widgets.py", "step_project_inventory.py")
    for name in critical:
        relative = "scripts/" + name
        path = project / relative
        expected_hash = manifest.get("files", {}).get(relative, {}).get("sha256")
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected_hash:
            raise RuntimeError(f"Installed code is missing or differs from its release manifest: {relative}. " + guidance)
    print(f"Installed release {expected}: {len(critical)} runtime file hashes verified.")
_check_installed_release(SCRIPTS.parent, "19.2.2")

previous_scripts = globals().get("_NIH_TRACKER_PREVIOUS_SCRIPTS")
if previous_scripts and previous_scripts in sys.path:
    sys.path.remove(previous_scripts)
if str(SCRIPTS) in sys.path:
    sys.path.remove(str(SCRIPTS))
sys.path.insert(0, str(SCRIPTS))
_NIH_TRACKER_PREVIOUS_SCRIPTS = str(SCRIPTS)
for module_name in list(sys.modules):
    if (
        module_name in {"tracker_gcp_backend", "evidence_store", "evidence_collector", "capability_assessment", "gcp_review", "gcp_workbench", "gcp_config", "tracker_cache_bridge", "gcp_review_queue", "tracker_claude_backend", "maintenance_widgets", "run_provenance", "code_provenance", "step_project_inventory", "common", "identity_hygiene", "interoperability_schema", "resource_profile_schema", "policy_context_schema", "repository_status_schema", "publisher_context_schema", "notebook_runtime", "review_widgets", "review_workflow", "automated_review", "verify_external_ids", "catalog_schema", "catalog_vocab", "catalog_claims", "catalog_graph", "catalog_resolver", "catalog_history", "catalog_exports"}
        or module_name.startswith("step_")
    ):
        sys.modules.pop(module_name, None)

from notebook_runtime import NotebookRuntime
RUNTIME = NotebookRuntime.create(
    notebook_name=NOTEBOOK_NAME,
    env="prod",
    enable_prod_writes=ENABLE_PROD_WRITES,
    roots=ROOTS,
)
common = RUNTIME.common
print(f"Ready: notebook={NOTEBOOK_NAME} | production | project={common.PROJECT}")
print(f"Debug session: {RUNTIME.debug_dir}")


### Reviewer identity and guided controls

This cell loads the named reviewer and initializes the compact guided controls. No provider API is called.


In [ ]:
import json
import os
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, clear_output, display

from automated_review import (
    EXPORT_RUN_ROOT,
    UPLOAD_RUN_ROOT,
    launch_ai_recommendation_triage,
    load_approvals,
    load_recommendations,
)
from review_widgets import launch_review_workbench
from review_workflow import (
    compute_review_status,
    pending_recommendations_for_human_approval,
)

RUNTIME.load_secrets(["NIH_TRACKER_REVIEWER"], quiet=True)
REVIEWER = os.getenv("NIH_TRACKER_REVIEWER", "").strip()
print("Reviewer identity:", REVIEWER if REVIEWER else "MISSING")
if not REVIEWER:
    print("Add NIH_TRACKER_REVIEWER to Colab secrets before saving approvals or decisions.")


## 2. Review dashboard

Refresh this dashboard after importing recommendations, saving approvals, or finalizing decisions. A green **COMPLETE** state means the review notebook is finished and the main pipeline is the next step. Evidence backlogs are reported separately and do not change this state. Queue labels use `pending | decided`; when all rows are decided, the dashboard states **No pending rows.**


In [ ]:
review_status_output = widgets.Output()
review_refresh_button = widgets.Button(
    description="Refresh review status",
    button_style="info",
    icon="refresh",
)
GUIDED_REVIEW_STATUS = None


def _status_color(state):
    return {
        "COMPLETE": "#137333",
        "REVIEW_REQUIRED": "#b06000",
        "APPROVAL_REQUIRED": "#b06000",
        "READY_TO_FINALIZE": "#8a3ffc",
    }.get(state, "#5f6368")


def refresh_guided_status(_button=None):
    global GUIDED_REVIEW_STATUS
    GUIDED_REVIEW_STATUS = compute_review_status(include_evidence=True)
    status = GUIDED_REVIEW_STATUS
    rec = status["recommendations"]
    ops = status["operational_adjudications"]
    queue_rows = "".join(
        f"<tr><td>{item['label']}</td><td>{item['pending_rows']}</td>"
        f"<td>{item['decided_rows']}</td><td><code>{item['run']}</code></td></tr>"
        for item in status["queues"]
    ) or "<tr><td colspan='4'>No current review queue found.</td></tr>"
    evidence = ", ".join(
        f"{item['domain']}: {item['rows']}"
        for item in status.get("non_gating_evidence_backlogs", [])
    ) or "none"
    covered = int(
        rec.get("recommendation_items_covered_by_current_decision", 0) or 0
    )
    html = f"""
    <div style='border:1px solid #dadce0;border-radius:8px;padding:14px'>
      <div style='font-size:20px;font-weight:700;color:{_status_color(status['state'])}'>
        {status['state']}
      </div>
      <div style='margin:6px 0 12px 0'>{status['next_action']}</div>
      <table style='border-collapse:collapse'>
        <tr><th style='text-align:left;padding-right:24px'>Imported items awaiting approval</th><td>{rec['review_items_awaiting_human_approval']}</td></tr>
        <tr><th style='text-align:left;padding-right:24px'>Later recommendations already covered by current decisions</th><td>{covered}</td></tr>
        <tr><th style='text-align:left;padding-right:24px'>Approved recommendations awaiting promotion</th><td>{rec['approved_recommendations_pending_promotion']}</td></tr>
        <tr><th style='text-align:left;padding-right:24px'>Identifier adjudications awaiting application</th><td>{ops['identifier']}</td></tr>
        <tr><th style='text-align:left;padding-right:24px'>Crosswalk adjudications awaiting application</th><td>{ops['crosswalk']}</td></tr>
        <tr><th style='text-align:left;padding-right:24px'>Card/name adjudications awaiting application</th><td>{ops['card_name']}</td></tr>
      </table>
      <h4 style='margin-bottom:4px'>Current queues</h4>
      <table style='border-collapse:collapse;width:100%'>
        <tr><th align='left'>Family</th><th align='left'>Pending</th><th align='left'>Decided</th><th align='left'>Source</th></tr>
        {queue_rows}
      </table>
      <div style='margin-top:10px;color:#5f6368'>Non-gating evidence backlogs: {evidence}</div>
    </div>
    """
    with review_status_output:
        clear_output(wait=True)
        display(HTML(html))
    callback = globals().get("_refresh_guided_controls")
    if callable(callback):
        callback()
    return status


review_refresh_button.on_click(refresh_guided_status)
display(widgets.VBox([review_refresh_button, review_status_output]))
refresh_guided_status()


## 3. Follow the current review action

This panel is state-aware. It shows recommendation approval when imported items need a human decision, candidate-queue controls when new rows need review, and a direct instruction to continue to finalization when only operational adjudications remain. Empty queue dropdowns are hidden.


In [ ]:
import datetime as dt
import shutil

manual_queue_widget = widgets.Dropdown(
    options=[],
    description="Manual queue",
    layout=widgets.Layout(width="1050px"),
)
ai_queue_widget = widgets.Dropdown(
    options=[],
    description="AI batch queue",
    layout=widgets.Layout(width="1050px"),
)
review_method_widget = widgets.Dropdown(
    options=[("ChatGPT web package", "chatgpt"), ("Perplexity web package", "perplexity")],
    value="chatgpt",
    description="Package",
    layout=widgets.Layout(width="380px"),
)
batch_size_widget = widgets.Dropdown(
    options=[10, 15, 20, 25, 40, 50],
    value=20,
    description="Items/batch",
    layout=widgets.Layout(width="240px"),
)
open_manual_button = widgets.Button(
    description="Open manual review",
    icon="edit",
    button_style="info",
)
export_button = widgets.Button(
    description="Create review batches",
    icon="archive",
    button_style="primary",
)
download_choice = widgets.Dropdown(
    options=[],
    description="Batch",
    disabled=True,
    layout=widgets.Layout(width="760px"),
)
download_button = widgets.Button(
    description="Download selected batch",
    icon="download",
    disabled=True,
)
saved_export_widget = widgets.Dropdown(
    options=[],
    description="Saved export",
    layout=widgets.Layout(width="760px"),
)
restore_export_button = widgets.Button(
    description="Restore download files after restart",
    icon="refresh",
)
review_action_output = widgets.Output()
guided_next_action_html = widgets.HTML()
open_triage_button = widgets.Button(
    description="Open recommendations for approval",
    icon="tasks",
    button_style="success",
)

AI_EXPORTABLE_FAMILIES = {"identifier", "rrid", "fairsharing", "crosswalk", "card"}


def _queue_options(*, ai_exportable_only=False):
    if not GUIDED_REVIEW_STATUS:
        return []
    rows = []
    for item in GUIDED_REVIEW_STATUS["queues"]:
        if int(item["pending_rows"]) <= 0:
            continue
        if ai_exportable_only and item["family"] not in AI_EXPORTABLE_FAMILIES:
            continue
        rows.append((
            f"{item['label']} | {item['pending_rows']} pending | {item['run']}",
            item["source_file"],
        ))
    return rows


def _set_widget_options(widget, options):
    previous = widget.value
    widget.options = options
    widget.disabled = not bool(options)
    values = {value for _, value in options}
    if previous in values:
        widget.value = previous
    elif options:
        widget.value = options[0][1]


def _refresh_guided_controls():
    manual_options = _queue_options(ai_exportable_only=False)
    ai_options = _queue_options(ai_exportable_only=True)
    _set_widget_options(manual_queue_widget, manual_options)
    _set_widget_options(ai_queue_widget, ai_options)
    open_manual_button.disabled = not bool(manual_options) or not bool(REVIEWER)
    export_button.disabled = not bool(ai_options)

    status = GUIDED_REVIEW_STATUS or {}
    state = status.get("state", "")
    rec = status.get("recommendations", {})
    pending_approval = int(rec.get("review_items_awaiting_human_approval", 0) or 0)
    pending_finalize = (
        int(rec.get("approved_recommendations_pending_promotion", 0) or 0)
        + int(status.get("operational_pending_rows", 0) or 0)
    )

    open_triage_button.disabled = pending_approval == 0 or not bool(REVIEWER)
    open_triage_button.description = (
        f"Open {pending_approval} recommendation"
        f"{'s' if pending_approval != 1 else ''} for approval"
    )
    open_triage_button.layout.display = "" if pending_approval else "none"

    show_candidate_controls = state == "REVIEW_REQUIRED" and bool(
        manual_options or ai_options
    )
    if "candidate_controls_box" in globals():
        candidate_controls_box.layout.display = (
            "" if show_candidate_controls else "none"
        )

    if state == "APPROVAL_REQUIRED":
        guided_next_action_html.value = (
            "<div style='border-left:4px solid #b06000;padding:10px 12px'>"
            f"<b>Next:</b> inspect and approve {pending_approval} imported "
            "recommendation item(s). Candidate queues are already decided, so "
            "no queue dropdown is shown.</div>"
        )
    elif state == "READY_TO_FINALIZE":
        guided_next_action_html.value = (
            "<div style='border-left:4px solid #8a3ffc;padding:10px 12px'>"
            f"<b>Next:</b> continue to Finalize all reviewed decisions. "
            f"{pending_finalize} reviewed decision row(s) remain.</div>"
        )
    elif state == "REVIEW_REQUIRED":
        guided_next_action_html.value = (
            "<div style='border-left:4px solid #b06000;padding:10px 12px'>"
            "<b>Next:</b> review the pending queue shown below.</div>"
        )
    elif state == "COMPLETE":
        guided_next_action_html.value = (
            "<div style='border-left:4px solid #137333;padding:10px 12px'>"
            "<b>Review complete.</b> Continue to the main pipeline.</div>"
        )
    else:
        guided_next_action_html.value = ""

    if "finalize_button" in globals():
        finalize_button.disabled = pending_approval > 0 or pending_finalize == 0


def _open_pending_triage(_button):
    with review_action_output:
        clear_output(wait=True)
        if not REVIEWER:
            print("Reviewer identity is missing.")
            return
        recommendations = load_recommendations()
        if recommendations.empty:
            print("No imported recommendations were found.")
            return
        approvals = load_approvals()
        pending = pending_recommendations_for_human_approval(
            recommendations, approvals
        )
        if pending.empty:
            print(
                "No imported review item requires approval. "
                "A current human adjudication may already cover the item."
            )
            refresh_guided_status()
            return
        print(f"Opening {pending['review_key'].nunique()} review item(s) awaiting approval.")
        globals()["GUIDED_AI_TRIAGE"] = launch_ai_recommendation_triage(
            recommendations=pending,
            reviewer=REVIEWER,
            page_size=10,
        )


def _open_manual(_button):
    with review_action_output:
        clear_output(wait=True)
        if not manual_queue_widget.value:
            print("No pending manual-review queue is selected.")
            return
        globals()["GUIDED_MANUAL_REVIEW"] = launch_review_workbench(
            manual_queue_widget.value,
            reviewer=REVIEWER,
            page_size=10,
            include_decided=False,
        )


def _saved_export_reports():
    return sorted(
        EXPORT_RUN_ROOT.glob("*/export_report.json"),
        key=lambda path: path.stat().st_mtime_ns,
        reverse=True,
    )


def _refresh_saved_exports(select_path=None):
    options = []
    for path in _saved_export_reports():
        try:
            report = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if int(report.get("item_count", 0) or 0) <= 0:
            continue
        options.append((
            f"{report.get('bundle_id', path.parent.name)} | {report.get('item_count', 0)} items",
            str(path),
        ))
    saved_export_widget.options = options
    saved_export_widget.disabled = not bool(options)
    restore_export_button.disabled = not bool(options)
    if select_path and str(select_path) in {value for _, value in options}:
        saved_export_widget.value = str(select_path)


def _restore_export(_button):
    with review_action_output:
        clear_output(wait=True)
        if not saved_export_widget.value:
            print("No saved export is selected.")
            return
        report = _activate_downloads(saved_export_widget.value)
        print(f"Restored {report.get('batch_count', 0)} batch(es) from Drive.")


def _new_export_report(previous_paths):
    reports = sorted(
        EXPORT_RUN_ROOT.glob("*/export_report.json"),
        key=lambda path: path.stat().st_mtime_ns,
        reverse=True,
    )
    return next((path for path in reports if str(path.resolve()) not in previous_paths), None)


def _activate_downloads(report_path):
    report = json.loads(Path(report_path).read_text(encoding="utf-8"))
    options = []
    for entry in report.get("download_files", []):
        path = Path(entry.get("path", ""))
        if path.is_file() and "All batches" not in entry.get("label", ""):
            options.append((f"{entry['label']} | {path.stat().st_size / 1024:.0f} KiB", str(path)))
    if not options:
        master = Path(report.get("master_zip", ""))
        if master.is_file():
            options = [(f"Complete package | {master.stat().st_size / 1024:.0f} KiB", str(master))]
    download_choice.options = options
    download_choice.disabled = not bool(options)
    download_button.disabled = not bool(options)
    if options:
        download_choice.value = options[0][1]
    return report


def _export_batches(_button):
    with review_action_output:
        clear_output(wait=True)
        if not ai_queue_widget.value:
            print("No pending card/name, identifier or crosswalk queue is available for AI batch export.")
            return
        before = {str(path.resolve()) for path in EXPORT_RUN_ROOT.glob("*/export_report.json")}
        result = RUNTIME.run_script(
            "step_export_ai_review.py",
            [
                "--source", str(ai_queue_widget.value),
                "--provider", review_method_widget.value,
                "--batch-size", str(batch_size_widget.value),
            ],
            label="export_guided_review_package",
            writes_cards=False,
        )
        if not result.ok:
            print(f"Export failed; inspect {result.log_path}")
            return
        report_path = _new_export_report(before)
        if report_path is None:
            print("No new package was created. The selected queue may already be fully decided.")
            refresh_guided_status()
            return
        report = _activate_downloads(report_path)
        _refresh_saved_exports(report_path)
        print(f"Created {report.get('batch_count', 0)} batch(es) for {report.get('item_count', 0)} item(s).")
        print(f"Durable export: {report_path.parent}")
        print("Download one batch, review it in the selected web service, then import the returned JSON below.")


def _download_batch(_button):
    path = Path(download_choice.value or "")
    if not path.is_file():
        with review_action_output:
            print(f"Missing batch file: {path}")
        return
    try:
        from google.colab import files
        files.download(str(path))
    except Exception as exc:
        with review_action_output:
            print(f"Browser download could not start: {exc}")
            print(f"Use the Colab Files pane: {path}")


open_triage_button.on_click(_open_pending_triage)
open_manual_button.on_click(_open_manual)
export_button.on_click(_export_batches)
download_button.on_click(_download_batch)
restore_export_button.on_click(_restore_export)
_refresh_saved_exports()

candidate_controls_box = widgets.VBox(
    [
        widgets.HTML("<b>Manual review</b> — all pending queue families"),
        widgets.HBox([manual_queue_widget, open_manual_button]),
        widgets.HTML(
            "<b>File-exchange batches</b> — card/name, identifier and crosswalk candidates"
        ),
        ai_queue_widget,
        widgets.HBox([review_method_widget, batch_size_widget, export_button]),
        widgets.HBox([saved_export_widget, restore_export_button]),
        widgets.HBox([download_choice, download_button]),
    ]
)

display(
    widgets.VBox(
        [
            guided_next_action_html,
            open_triage_button,
            candidate_controls_box,
            review_action_output,
        ]
    )
)
_refresh_guided_controls()


## 4. Import returned files when needed

Use this section only after completing a ChatGPT/Perplexity web batch. **Import returned files** saves selected JSON/JSONL/ZIP files to Drive before validation and runs the strict source/hash/target checks. The state-aware panel above then exposes the human-approval action. Importing does not change cards.


In [ ]:
import_button = widgets.Button(
    description="Import returned files",
    icon="upload",
    button_style="primary",
)
saved_upload_widget = widgets.Dropdown(
    options=[],
    description="Saved uploads",
    layout=widgets.Layout(width="760px"),
)
refresh_uploads_button = widgets.Button(
    description="Refresh saved uploads",
    icon="refresh",
)
import_saved_button = widgets.Button(
    description="Import selected saved upload",
    icon="check",
)
import_triage_output = widgets.Output()


def _refresh_persisted_uploads(_button=None, select_path=None):
    options = []
    if UPLOAD_RUN_ROOT.is_dir():
        for directory in sorted(
            (path for path in UPLOAD_RUN_ROOT.iterdir() if path.is_dir()),
            key=lambda path: path.stat().st_mtime_ns,
            reverse=True,
        ):
            files_found = [
                path for path in directory.iterdir()
                if path.is_file() and path.suffix.lower() in {".json", ".jsonl", ".zip"}
            ]
            if files_found:
                options.append((f"{directory.name} | {len(files_found)} file(s)", str(directory)))
    saved_upload_widget.options = options
    saved_upload_widget.disabled = not bool(options)
    import_saved_button.disabled = not bool(options)
    if select_path and str(select_path) in {value for _, value in options}:
        saved_upload_widget.value = str(select_path)


def _persist_uploads(uploaded):
    import hashlib
    signature = hashlib.sha256()
    for name, payload in sorted(uploaded.items()):
        signature.update(Path(name).name.encode("utf-8"))
        signature.update(hashlib.sha256(payload).digest())
    stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    destination = UPLOAD_RUN_ROOT / f"{stamp}-{signature.hexdigest()[:12]}"
    destination.mkdir(parents=True, exist_ok=False)
    paths = []
    for name, payload in uploaded.items():
        path = destination / Path(name).name
        common.atomic_write_bytes(path, payload)
        paths.append(path)
    return destination, paths


def _import_paths(paths, destination):
    failures = 0
    for index, path in enumerate(paths, start=1):
        print(f"[{index}/{len(paths)}] {path.name}")
        result = RUNTIME.run_script(
            "step_import_ai_review.py",
            ["--response", str(path), "--require-complete-batches"],
            label=f"import_guided_review_{path.name}",
            writes_cards=False,
        )
        failures += int(not result.ok)
    if failures:
        print(f"{failures} file(s) failed validation. Originals remain at {destination}.")
    else:
        print("All selected files passed validation and entered the recommendation quarantine.")
    refresh_guided_status()


def _import_saved(_button):
    with import_triage_output:
        clear_output(wait=True)
        if not saved_upload_widget.value:
            print("No saved upload set is selected.")
            return
        destination = Path(saved_upload_widget.value)
        paths = sorted(
            path for path in destination.iterdir()
            if path.is_file() and path.suffix.lower() in {".json", ".jsonl", ".zip"}
        )
        _import_paths(paths, destination)


def _import_returned(_button):
    with import_triage_output:
        clear_output(wait=True)
        try:
            from google.colab import files
            selected = files.upload()
        except Exception as exc:
            print(f"Browser upload is unavailable: {exc}")
            return
        if not selected:
            print("No file selected.")
            return
        uploaded = {name: bytes(payload) for name, payload in selected.items()}
        destination, paths = _persist_uploads(uploaded)
        print(f"Selected and saved to Drive: {destination}")
        _refresh_persisted_uploads(select_path=destination)
        _import_paths(paths, destination)



import_button.on_click(_import_returned)
refresh_uploads_button.on_click(_refresh_persisted_uploads)
import_saved_button.on_click(_import_saved)
_refresh_persisted_uploads()
display(
    widgets.VBox(
        [
            widgets.HBox([import_button]),
            widgets.HBox([saved_upload_widget, refresh_uploads_button, import_saved_button]),
            import_triage_output,
        ]
    )
)
_refresh_guided_controls()


## 5. Finalize all reviewed decisions

This is the only normal application control. **Validate** previews every pending reviewed family. **Apply all** promotes approved AI recommendations, revalidates the materialized operational decisions, and applies only nonempty families. Empty card/name, crosswalk, and identifier families are skipped rather than launched as repeated no-op subprocesses.


In [ ]:
finalize_action_widget = widgets.ToggleButtons(
    options=[("Validate", "validate"), ("Apply all", "apply")],
    value="validate",
    description="Action",
)
finalize_button = widgets.Button(
    description="Run guided finalization",
    button_style="warning",
    icon="check-circle",
)
finalize_output = widgets.Output()


def _run_finalize(_button):
    finalize_button.disabled = True
    try:
      with finalize_output:
        clear_output(wait=True)
        status = GUIDED_REVIEW_STATUS or {}
        pending_approval = int(status.get("recommendations", {}).get("review_items_awaiting_human_approval", 0) or 0)
        if pending_approval:
            print(f"Finalization is disabled: {pending_approval} review item(s) still require human approval.")
            return
        action = finalize_action_widget.value
        if action == "apply" and common.ENV == "prod" and not ENABLE_PROD_WRITES:
            print("Production application requires ENABLE_PROD_WRITES in the setup cell.")
            return
        args = ["--action", action]
        if action == "apply":
            args.extend(["--confirm", "FINALIZE_REVIEW_WORKFLOW"])
        result = RUNTIME.run_script(
            "step_review_workflow.py",
            args,
            label=f"guided_review_{action}",
            writes_cards=action == "apply",
        )
        if result.ok:
            print("Guided review workflow completed.")
        else:
            print(f"Guided review workflow stopped; inspect {result.log_path}")
        refresh_guided_status()
    finally:
        _refresh_guided_controls()


finalize_button.on_click(_run_finalize)
display(widgets.VBox([finalize_action_widget, finalize_button, finalize_output]))
_refresh_guided_controls()


## 6. Advanced diagnostics (optional)

            Open this section only for historical queues, targeted family-level troubleshooting, or evidence-curation planning. Evidence gaps are descriptive backlogs—not adjudication queues—and are summarized without automatically rendering thousand-row tables. The current files are `interoperability_gaps.csv` with `templates/interoperability_curated_template.csv`, and `resource_profile_gaps.csv` with `templates/resource_profile_curated_template.csv`.

The former **Bulk finalize all human-approved AI recommendations** / **Promote and apply all approved AI recommendations** sequence is now the single guided finalization action. Internally, guarded family scripts remain `step_apply_verify_adjudications.py` (using `--bulk-safe`), `step_apply_crosswalk_adjudications.py`, and `step_promote_review.py`. Crosswalk `DEFER` rows remain reviewed nonblocking deferrals; an empty family remains a valid no-op; no unapplied crosswalk decisions are processed.


In [ ]:
advanced_checkbox = widgets.Checkbox(
    value=False,
    description="Show advanced diagnostics",
    indent=False,
)
advanced_domain_widget = widgets.Dropdown(
    options=[("Interoperability gaps", "interoperability"), ("Resource-profile gaps", "resource_profile")],
    description="Evidence",
    disabled=True,
    layout=widgets.Layout(width="440px"),
)
advanced_rows_widget = widgets.Dropdown(
    options=[25, 50, 100, 200],
    value=50,
    description="Max rows",
    disabled=True,
)
advanced_open_button = widgets.Button(
    description="Open selected evidence report",
    icon="table",
    disabled=True,
)
advanced_output = widgets.Output()


def _toggle_advanced(change=None):
    enabled = bool(advanced_checkbox.value)
    advanced_domain_widget.disabled = not enabled
    advanced_rows_widget.disabled = not enabled
    advanced_open_button.disabled = not enabled
    with advanced_output:
        clear_output(wait=True)
        if not enabled:
            print("Advanced diagnostics are hidden.")
            return
        status = compute_review_status(include_evidence=True)
        print("Current queue inventory:")
        display(pd.DataFrame(status["queues"]))
        print("Non-gating evidence backlog summary:")
        display(pd.DataFrame(status.get("non_gating_evidence_backlogs", [])))
        print("Family-specific CLI scripts remain available under scripts/ for targeted forensic use.")


def _open_evidence(_button):
    with advanced_output:
        domain = advanced_domain_widget.value
        status = compute_review_status(include_evidence=True)
        match = next(
            (item for item in status.get("non_gating_evidence_backlogs", []) if item["domain"] == domain),
            None,
        )
        if not match or not match.get("source_file"):
            print(f"No current {domain} evidence report was found.")
            return
        path = Path(match["source_file"])
        frame = pd.read_csv(path, dtype=str, keep_default_na=False).fillna("")
        if "status" in frame.columns:
            frame = frame.loc[~frame["status"].str.lower().isin(["resolved", "accepted", "not_applicable"])]
        limit = int(advanced_rows_widget.value)
        print(f"{path} | showing {min(limit, len(frame))} of {len(frame)} unresolved row(s)")
        display(frame.head(limit))


advanced_checkbox.observe(_toggle_advanced, names="value")
advanced_open_button.on_click(_open_evidence)
display(
    widgets.VBox(
        [
            advanced_checkbox,
            widgets.HBox([advanced_domain_widget, advanced_rows_widget]),
            advanced_open_button,
            advanced_output,
        ]
    )
)
_toggle_advanced()


## 7. Session audit

This read-only summary lists only subprocesses actually launched during the guided session. When the dashboard is complete, proceed to the main pipeline notebook.


In [ ]:
print(f"Debug directory: {RUNTIME.debug_dir}")
if RUNTIME.run_results():
    display(pd.DataFrame(RUNTIME.run_results()))
RUNTIME.write_session_summary()
refresh_guided_status()


## Optional GCP evidence and assessment (v19.1.1)

This extends the v19.0.2 guided workflow. Enable it after installing this release.
Gemini recommendations, with an optional independent Claude assessment, enter
the existing **Review → Approve → Finalize** flow. Card/name candidates are now
included. The Gemini review tab has a resumable queue plan; see
`GCP_SEMIAUTOMATED_REVIEW.md` for the 132-card plus one-identifier backlog.
The A–D panel creates a separate evidence inventory and named review record;
it does not automatically publish claims or change cards.

Add `GCP_ID` to Colab Secrets and enable notebook access. The project uses Google
sign-in (ADC), not a Gemini API key. `NIH_TRACKER_REVIEWER` remains the reviewer
identity. Configure the bucket, BigQuery dataset, collector contact and approved
source scope in the Settings tab. Start with five resources.

Cloud writes, collection, Gemini calls and interactive Google Search each require
their own explicit action. Durable cache search uses independently captured sources.
Google-grounded research results are not evidence-cache inputs or crawler seeds.


In [ ]:
# NIH_TRACKER_GCP_EXTENSION_V19_1
ENABLE_GCP_TOOLS = False # @param {type:"boolean"}
if ENABLE_GCP_TOOLS:
    import subprocess
    import sys
    import os
    from pathlib import Path
    requirement = Path(common.PROJECT) / 'requirements-gcp.txt'
    if not requirement.is_file():
        raise FileNotFoundError('Install the v19.1.1 release with the maintenance notebook first.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirement)], check=True)
    from importlib import metadata
    restarted_modules = []
    for module_name, package_name in [('pandas','pandas'), ('yaml','PyYAML'), ('requests','requests'),
                                      ('nbformat','nbformat'), ('ipywidgets','ipywidgets')]:
        loaded = sys.modules.get(module_name)
        if loaded is not None and getattr(loaded, '__version__', metadata.version(package_name)) != metadata.version(package_name):
            restarted_modules.append(package_name)
    if restarted_modules:
        raise RuntimeError('Dependencies changed (' + ', '.join(restarted_modules) + '). Restart the Colab runtime and rerun the notebook from the top.')
    from google.colab import auth, userdata
    try:
        os.environ['GCP_ID'] = userdata.get('GCP_ID').strip()
    except Exception:
        raise ValueError('Add GCP_ID to Colab Secrets and enable notebook access.') from None
    from gcp_config import load
    import tracker_gcp_backend
    tracker_gcp_backend.project_id(load(common.PROJECT))
    auth.authenticate_user()
    from gcp_workbench import launch
    GCP_PANEL = launch(common.PROJECT, reviewer=os.getenv('NIH_TRACKER_REVIEWER',''),
                       on_recommendations=globals().get('refresh_guided_status'))
    display(GCP_PANEL)
else:
    print('Optional GCP tools disabled; existing guided review remains available.')
